In [1]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 07. Backpropagation text — text text text ⭐

> **Learning Goals**
> - text text text Trainingtext text text textdegreestext
> - Calculation Graph(computational graph)text text text Derivativetext Calculationtext
> - Autogradtext text text text
> - Backpropagationtext CPU vs GPU Speed text text

text text text text text. PyTorchtext `autograd`text text text text text text text.

## 7.1 text Backpropagationtext

text text $\mathcal{L}(\theta)$text text text $\nabla_\theta \mathcal{L}$text text.

**textDirection Numerical Differentiation(forward-mode AD)**: text text text text Forward Pass.
- text $N$text → $N$text Forward Pass. text.

**textDirection text text(reverse-mode AD, Backpropagation)**: text text Backpropagationtext text text text Calculation.
- $N$text text → 1text Forward Pass + 1text Backpropagation. textdegreestext text.


## 7.2 text text text

$$\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x)$$

text text: $y = f(u_1, u_2, \ldots, u_n)$, $u_i = g_i(x)$text text

$$\frac{dy}{dx} = \sum_i \frac{\partial f}{\partial u_i} \cdot \frac{du_i}{dx}$$

text Backpropagationtext text text.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# text text Function: y = f(g(h(x))) = (h(x)^2 + 1) * 2, h(x) = x + 1
# = ((x+1)^2 + 1) * 2
def f(x): return ((x + 1)**2 + 1) * 2
def df_dx(x): return 2 * (x + 1) * 2 * 1  # text text

# Numerical Differentiationtext Verification
def numerical_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 3.0
print(f"f(x) = ((x+1)^2 + 1) * 2")
print(f"f'({x0}) Analytic (text text): {df_dx(x0)}")
print(f"f'({x0}) text: {numerical_diff(f, x0):.6f}")
print(f"Error: {abs(df_dx(x0) - numerical_diff(f, x0)):.2e}")


f(x) = ((x+1)^2 + 1) * 2
f'(3.0) Analytic (text text): 16.0
f'(3.0) text: 16.000000
Error: 2.50e-10


## 7.3 Calculation Graph

text Functiontext text text(text, text, text text)text Graphtext text.

text: $\mathcal{L} = (\sigma(w \cdot x + b) - y)^2$

```
x ──┐
    ├─→ (×) ──→ (+) ──→ (σ) ──→ (-) ──→ (²) ──→ L
w ──┘         ↑              ↑
              b              y
```

text text **text Derivative(local gradient)**text Calculationtext text. Backpropagationtext text text text text.


In [3]:
# text Autograd text text
class Tensor:
    """text text Differential Tensor."""
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        self.data = np.asarray(data, dtype=float)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor({self.data}, grad={self.grad})"

    @property
    def shape(self):
        return self.data.shape

    # Operation text
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, _children=(self, other), _op='+')

        def _backward():
            # Additiontext text Derivative = 1
            self.grad = (self.grad if self.grad is not None else 0) + out.grad
            other.grad = (other.grad if other.grad is not None else 0) + out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, _children=(self, other), _op='*')

        def _backward():
            # Multiplicationtext text Derivative: d/da(a*b) = b, d/db(a*b) = a
            self.grad = (self.grad if self.grad is not None else 0) + other.data * out.grad
            other.grad = (other.grad if other.grad is not None else 0) + self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), _children=(self,), _op='relu')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1 / (1 + np.exp(-self.data))
        out = Tensor(s, _children=(self,), _op='sigmoid')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + s * (1 - s) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), _children=(self,), _op='sum')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # text Alignment
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = np.ones_like(self.data)  # dL/dL = 1
        for v in reversed(topo):
            v._backward()

# Test: f(x, y) = (x + y) * (x - y) = x^2 - y^2 (text x*x - y*y)
x = Tensor(3.0, requires_grad=True)
y = Tensor(4.0, requires_grad=True)
# z = x*x - y*y  (subtraction = add negative; Implementation text text text)
a = x * x  # x^2
b = y * y  # y^2
z = a + (b * Tensor(-1.0))  # x^2 - y^2
z.backward()

print(f"x = {x.data}, y = {y.data}")
print(f"z = x^2 - y^2 = {z.data}")
print(f"dz/dx = {x.grad} (Theory 2x = {2*x.data})")
print(f"dz/dy = {y.grad} (Theory -2y = {-2*y.data})")


x = 3.0, y = 4.0
z = x^2 - y^2 = -7.0
dz/dx = 6.0 (Theory 2x = 6.0)
dz/dy = -8.0 (Theory -2y = -8.0)


## 7.4 text Backpropagation textdegrees

text: $\mathbf{y} = W\mathbf{x} + \mathbf{b}$

text $\mathcal{L}$text text text:

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \cdot \mathbf{x}^\top = \boldsymbol{\delta} \, \mathbf{x}^\top$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = W^\top \boldsymbol{\delta}$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \boldsymbol{\delta}$$

text $\boldsymbol{\delta} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}}$text "text text".


In [4]:
# text Backpropagation Verification (PyTorchtext Comparison)
import torch

# text
x_np = np.random.randn(3, 4)  # batch=3, in=4
W_np = np.random.randn(4, 5)  # in=4, out=5
b_np = np.random.randn(5)

# NumPytext Forward Pass + Backpropagation
def linear_forward(x, W, b):
    return x @ W + b

def linear_backward(x, W, delta):
    """delta: dL/dy (N, out)"""
    dW = x.T @ delta          # (in, out)
    db = delta.sum(axis=0)    # (out,)
    dx = delta @ W.T          # (N, in)
    return dW, db, dx

# text delta (dL/dy)
delta_np = np.random.randn(3, 5)
y_np = linear_forward(x_np, W_np, b_np)
dW_np, db_np, dx_np = linear_backward(x_np, W_np, delta_np)

# PyTorchtext Validation
x_t = torch.tensor(x_np, requires_grad=True)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
y_t = x_t @ W_t + b_t
y_t.backward(torch.tensor(delta_np))

print("NumPy vs PyTorch Backpropagation Comparison:")
print(f"  dW Maximum Error: {np.max(np.abs(dW_np - W_t.grad.numpy())):.2e}")
print(f"  db Maximum Error: {np.max(np.abs(db_np - b_t.grad.numpy())):.2e}")
print(f"  dx Maximum Error: {np.max(np.abs(dx_np - x_t.grad.numpy())):.2e}")
print("\n=> NumPy text Implementationtext PyTorch autogradtext text text!")


NumPy vs PyTorch Backpropagation Comparison:
  dW Maximum Error: 0.00e+00
  db Maximum Error: 0.00e+00
  dx Maximum Error: 8.88e-16

=> NumPy text Implementationtext PyTorch autogradtext text text!


## 7.5 MLP Backpropagation text textdegrees

2text MLP: $\mathbf{o} = W_2 \sigma(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$

text $\mathcal{L} = \|\mathbf{o} - \mathbf{y}\|^2$ (MSE)

Backpropagation:
1. $\frac{\partial \mathcal{L}}{\partial \mathbf{o}} = 2(\mathbf{o} - \mathbf{y})$
2. $\frac{\partial \mathcal{L}}{\partial W_2} = \frac{\partial \mathcal{L}}{\partial \mathbf{o}} \mathbf{h}^\top$
3. $\frac{\partial \mathcal{L}}{\partial \mathbf{h}} = W_2^\top \frac{\partial \mathcal{L}}{\partial \mathbf{o}}$
4. $\frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}} \odot \sigma'(\mathbf{z}_1)$
5. $\frac{\partial \mathcal{L}}{\partial W_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} \mathbf{x}^\top$

text: **text text text Derivative × text text**text text.


In [5]:
# MLP Backpropagation Numerical Differentiationtext Verification
# f(W1, b1, W2, b2) = ||W2 @ sigmoid(W1 @ x + b1) + b2 - y||^2

def mlp_loss_and_grads(x, W1, b1, W2, b2, y):
    """Forward Pass + Backward Pass (Vector Input, text Sample)."""
    # Forward Pass
    z1 = W1 @ x + b1
    h = 1 / (1 + np.exp(-z1))  # sigmoid
    o = W2 @ h + b2
    diff = o - y
    loss = np.sum(diff**2)
    # Backward Pass
    do = 2 * diff
    dW2 = np.outer(do, h)
    db2 = do
    dh = W2.T @ do
    dz1 = dh * h * (1 - h)
    dW1 = np.outer(dz1, x)
    db1 = dz1
    return loss, dW1, db1, dW2, db2

# text Differential
def numerical_grad(f, param, h=1e-5):
    """ftext losstext text Function. paramtext text text Gradient."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = param[idx]
        param[idx] = old + h
        loss_plus = f()
        param[idx] = old - h
        loss_minus = f()
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad

# text MLPtext Validation
np.random.seed(0)
x = np.random.randn(3)
y = np.random.randn(2)
W1 = np.random.randn(4, 3)
b1 = np.random.randn(4)
W2 = np.random.randn(2, 4)
b2 = np.random.randn(2)

loss, dW1, db1, dW2, db2 = mlp_loss_and_grads(x, W1, b1, W2, b2, y)

# text Gradient
loss_fn = lambda: mlp_loss_and_grads(x, W1, b1, W2, b2, y)[0]
dW1_num = numerical_grad(loss_fn, W1)
dW2_num = numerical_grad(loss_fn, W2)

print(f"dW1 text Error: {np.max(np.abs(dW1 - dW1_num)):.2e}")
print(f"dW2 Maximum Error: {np.max(np.abs(dW2 - dW2_num)):.2e}")
print("=> Analytic Backward Passtext text Differentialtext text!")


dW1 text Error: 4.81e-11
dW2 Maximum Error: 2.04e-11
=> Analytic Backward Passtext text Differentialtext text!


## 7.6 text text/text

text Derivative $\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$.

text text Backpropagation text text text Valuetext text text text text text → **text text(vanishing gradient)**.

text:
- ReLU text text (Derivativetext 0 text 1)
- text text text (He, Xavier)
- Residual connection (text text)
- BatchNorm/LayerNorm


In [6]:
# text text text
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

# text text text text
np.random.seed(0)
depths = [5, 10, 20, 50, 100]
print(f"{'Depth':>6} | {'Mean |grad|':>15} | {'Minimum |grad|':>15}")
print("-" * 45)
for d in depths:
    x = np.random.randn(100)
    grad = np.ones(100)  # dL/dy
    for _ in range(d):
        # Backward Pass text Layer
        z = np.random.randn(100)
        grad = grad * sigmoid_deriv(z)
    print(f"{d:>6} | {np.mean(np.abs(grad)):>15.2e} | {np.min(np.abs(grad)):>15.2e}")

print("\n=> Depthtext text Gradienttext text text (text)!")
print("ReLUtext textPlane Derivativetext 0 text 1text text Problemtext text text.")


 Depth |     Mean |grad| |  Minimum |grad|
---------------------------------------------
     5 |        3.71e-04 |        2.23e-05
    10 |        1.51e-07 |        1.51e-08
    20 |        2.49e-14 |        5.22e-16
    50 |        6.93e-35 |        2.79e-38
   100 |        2.70e-69 |        2.82e-73

=> Depthtext text Gradienttext text text (text)!
ReLUtext textPlane Derivativetext 0 text 1text text Problemtext text text.


## 7.7 [CPU/GPU Benchmark ②] Backpropagation Time Comparison

By Model Sizetext Forward Pass+Backpropagation Timetext CPU vs GPUtext Comparisontext.


In [7]:
# Backpropagation Benchmark
import torch
import time
from llm_math.bench import time_fn, format_results_table

def make_mlp_and_run(input_dim, hidden, output_dim, n_layers, batch_size=32, device='cpu'):
    """MLPtext text text Step Forward Pass+Backward Pass."""
    dims = [input_dim] + [hidden] * (n_layers - 1) + [output_dim]
    layers = [torch.nn.Linear(dims[i], dims[i+1]) for i in range(len(dims) - 1)]
    model = torch.nn.Sequential(*layers).to(device)
    x = torch.randn(batch_size, input_dim, device=device)
    y = torch.randn(batch_size, output_dim, device=device)
    loss_fn = torch.nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.01)

    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

# Model Sizetext Comparison
configs = [
    ('small',  100, 64, 10, 3),
    ('medium', 100, 256, 10, 5),
    ('large',  100, 512, 10, 10),
    ('xlarge', 100, 1024, 10, 20),
]

print(f"{'Config':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for name, in_d, hid, out_d, nl in configs:
    step_cpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cpu')
    res_cpu = time_fn(step_cpu, device='cpu', warmup=2, repeat=5)
    if torch.cuda.is_available():
        step_gpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cuda')
        res_gpu = time_fn(step_gpu, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU text. Colabtext T4 GPU text textPlane text text text text.")
    print("  text text Model(xlarge)text CPUtext text text, GPUtext text mstext text.")


  Config |     CPU (ms) |     GPU (ms) |    Speedup
--------------------------------------------------


   small |        0.341 |          N/A |        N/A
  medium |        3.122 |          N/A |        N/A
   large |       13.843 |          N/A |        N/A


  xlarge |       90.379 |          N/A |        N/A

⚠ GPU text. Colabtext T4 GPU text textPlane text text text text.
  text text Model(xlarge)text CPUtext text text, GPUtext text mstext text.


## 7.8 Key Takeaways

| text | text | text |
|---|---|---|
| text text | $\frac{d}{dx}f(g(x)) = f'(g)g'(x)$ | textFunction text |
| Calculation Graph | — | textFunctiontext text text text |
| text Derivative | — | text text text text Output text |
| Backpropagation | $\nabla_\theta \mathcal{L}$ | 1text Backpropagationtext text text |
| text Backpropagation | $\partial\mathcal{L}/\partial W = \delta \mathbf{x}^\top$ | text text |

## Exercises

1. text Autograd `Tensor` text `__sub__`, `__truediv__`, `matmul` text text.
2. text Autogradtext $f(x, y) = (x + y)^2 - xy$text $\partial f/\partial x, \partial f/\partial y$text Calculationtext Numerical Differentiationtext Verificationtext.
3. PyTorchtext 3text MLPtext text, text text Backpropagation text `W1.grad`, `W2.grad`text Outputtext.
4. text 1, 5, 10, 20text MLPtext gradient normtext Comparisontext text text text text.
5. He text text text(text `randn`)text Comparisontext text MLP Trainingtext text text text.

> Solutions: `solutions/ch07_solutions.ipynb`
